In [1]:
%cd ../..

/home/philippe/MolGenDocking


In [2]:
import json

from pathlib import Path
from tqdm.auto import tqdm

In [3]:
RES_DIR = Path("MolGenOutput/property_prediction")
REFERENCE = RES_DIR / "Qwen3-30B-A3B-Thinking-2507"
TO_SPREAD = RES_DIR / "MiniMax-M2"
assert REFERENCE.exists() and TO_SPREAD.exists(), "Reference and to spread directories must exist."

In [4]:
prompt_id_dict = {}
for i in tqdm(range(10)):
    file = REFERENCE / f"out_{i}.jsonl"
    if not file.exists():
        break
    prompt_id_dict[i] = []
    with open(file, "r") as f:
        for line in f:
            row = json.loads(line)
            prompt_id = row["metadata"]["prompt_id"]
            prompt_id_dict[i].append(prompt_id)


  0%|          | 0/10 [00:00<?, ?it/s]

In [5]:
new_dir = RES_DIR / "MiniMax-M2-spread"
new_dir.mkdir(exist_ok=True)

# Step 1: Gather all completions from the Spread directory
idx_completions = {i: [] for i in prompt_id_dict}
for file in tqdm(TO_SPREAD.glob("*.jsonl")):
    if not file.exists():
        break
    with open(file, "r") as f:
        for line in f:
            row = json.loads(line)
            prompt_id = row["metadata"]["prompt_id"]
            # Find the prompt id in prompt_id_dict
            idx = None
            for idx_test in prompt_id_dict:
                if prompt_id in prompt_id_dict[idx_test]:
                    idx = idx_test
                    break
            assert idx is not None, "PROMPT_ID NOT FOUND"
            idx_completions[idx].append(row)


0it [00:00, ?it/s]

In [6]:
# Create files
for idx, completions in tqdm(idx_completions.items()):
    if len(completions) > 1:
        out_file = new_dir / f"out_{idx}.jsonl"
        with open(out_file, "w") as f:
            for completion in completions:
                f.write(json.dumps(completion) + "\n")

  0%|          | 0/10 [00:00<?, ?it/s]